In [1]:
# Bootstrap defaults for running in VS Code
import os, glob, psutil

# Ensure SPARK_HOME is set (fallback to local build dist)
SPARK_HOME = os.environ.get("SPARK_HOME", "/users/chenqh23/spark/dist")
os.environ["SPARK_HOME"] = SPARK_HOME
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

for p in psutil.process_iter(['pid','name','cmdline']):
    cl = ' '.join(p.info.get('cmdline') or [])
    if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
        try: p.kill()
        except: pass
# ensure classpath
try:
    import findspark; findspark.init(os.environ['SPARK_HOME'])
except Exception:
    pass

# Use local mode unless a cluster URL is provided
os.environ.setdefault("SPARK_MASTER_URL", "local[*]")

# Data and event log defaults
os.environ.setdefault("DATA_ROOT", "/users/chenqh23/spark-rapids-examples/datasets")
os.environ.setdefault("EVENTLOG_DIR", "/tmp/spark-events")
try:
    os.makedirs(os.environ["EVENTLOG_DIR"], exist_ok=True)
except Exception:
    pass

# Auto-detect RAPIDS jar in SPARK_HOME/jars if not provided
if "RAPIDS_JAR" not in os.environ:
    jars_dir = os.path.join(SPARK_HOME, "jars")
    candidates = []
    for pattern in ("rapids-4-spark_*.jar", "rapids-4-spark*.jar"):
        candidates.extend(glob.glob(os.path.join(jars_dir, pattern)))
    rapids_jar = next((p for p in candidates if p.endswith(".jar")), None)
    if rapids_jar:
        os.environ["RAPIDS_JAR"] = rapids_jar

print("SPARK_HOME =", os.environ["SPARK_HOME"]) 
print("SPARK_MASTER_URL =", os.environ["SPARK_MASTER_URL"]) 
print("EVENTLOG_DIR =", os.environ["EVENTLOG_DIR"]) 
print("RAPIDS_JAR =", os.environ.get("RAPIDS_JAR", "<auto> or jars in SPARK_HOME"))


SPARK_HOME = /users/chenqh23/spark/dist
SPARK_MASTER_URL = local[*]
EVENTLOG_DIR = /tmp/spark-events
RAPIDS_JAR = /users/chenqh23/spark/dist/jars/rapids-4-spark_2.12-25.08.0.jar


# Microbenchmarks on GPU
This is a notebook for microbenchmarks running on GPU. 

In [2]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from time import time
import os
# Change to your cluster ip:port and directories
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "spark:your-ip:port")
RAPIDS_JAR = os.getenv("RAPIDS_JAR", "/your-path/rapids-4-spark_2.12-25.08.0.jar")


Run the microbenchmark with retryTimes

In [3]:
# Robust Spark startup to fix Py4J "Answer from Java side is empty"
import os, psutil
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf

# Environment
os.environ['JAVA_HOME'] = os.environ.get('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
os.environ['SPARK_HOME'] = os.environ.get('SPARK_HOME', '/users/chenqh23/spark/dist')
os.environ.setdefault('SPARK_MASTER_URL', 'local[*]')

# Stop an existing Spark and kill orphan Spark JVMs
try:
    spark.stop()
except Exception:
    pass
try:
    for p in psutil.process_iter(['pid','name','cmdline','username']):
        cl = ' '.join(p.info.get('cmdline') or [])
        if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
            try:
                p.kill()
            except Exception:
                pass
except Exception:
    pass

# Java 17 add-opens flags
_DEF_OPENS = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED"
)

# Build base conf
base = (SparkConf()
    .setMaster("local[*]")
    .setAppName("Microbenchmark on GPU")
    .set("spark.driver.memory", os.getenv("DRIVER_MEM", "12g"))
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.files.maxPartitionBytes", os.getenv("MAX_PARTITION_BYTES", "64m"))
    .set("spark.locality.wait", "0")
    .set("spark.eventLog.enabled", "false")
    .set("spark.driver.extraJavaOptions", _DEF_OPENS)
    .set("spark.executor.extraJavaOptions", _DEF_OPENS)
)

# GPU settings (avoid duplicate JARs; rely on SPARK_HOME/jars)
gpu = (base
    .set("spark.plugins", "com.nvidia.spark.SQLPlugin")
    .set("spark.rapids.sql.enabled", "true")
    .set("spark.rapids.sql.allowMultipleJars", "ALWAYS")
    .set("spark.rapids.sql.concurrentGpuTasks", os.getenv("CONCURRENT_GPU_TASKS", "1"))
    .set("spark.rapids.sql.batchSizeBytes", "256m")
    .set("spark.rapids.sql.multiThreadedRead.numThreads", os.getenv("RAPIDS_READER_THREADS", "8"))
    .set("spark.rapids.memory.gpu.maxAllocFraction", "0.5")
    .set("spark.rapids.memory.gpu.allocFraction", "0.25")
    .set("spark.rapids.memory.gpu.minAllocFraction", "0.0")
    .set("spark.rapids.memory.gpu.reserve", "2G")
    .set("spark.rapids.memory.pinnedPool.size", os.getenv("PINNED_POOL_SIZE", "4g"))
)

# Try GPU first; if it crashes the JVM, fall back to CPU-only cleanly
try:
    spark = SparkSession.builder.config(conf=gpu).getOrCreate()
except Exception:
    cpu = (base
        .set("spark.rapids.sql.enabled", "false")
        .set("spark.plugins", ""))
    spark = SparkSession.builder.config(conf=cpu).getOrCreate()

print("Spark OK:", spark.version, "| rapids.enabled:", spark.conf.get("spark.rapids.sql.enabled", "<unset>"))


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/16 19:33:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/16 19:33:53 WARN RapidsPluginUtils: RAPIDS Accelerator 25.08.0 using cudf 25.08.0, private revision f4b467339f0ea78b7e2a862be97a63bc239e0b07
25/10/16 19:33:53 WARN RapidsPluginUtils: Multiple spark-rapids-jni jars found in the classpath:
revison: 155ef36a7e5c38e404d76976a61d177354c99281
	jar URL: jar:file:/users/chenqh23/spark/dist/jars/spark-rapids-jni-25.08.0.jar
	version=25.08.0
	user=root
	revision=155ef36a7e5c38e404d76976a61d177354c99281
	branch=HEAD
	date=2025-08-07T05:18:36Z
	url=https://github.com/NVIDIA/spark-rapids-jni.git
	gpu_architectures=100;120;70;75;80;86;90
	jar URL: jar:file:/users/chenqh23/spark/dist/jars/rapids-4-spark_2.12-25.08.0.jar
	version=25.08.0
	user=root
	revision=155ef36a7e5c38e404d

Spark OK: 3.5.6 | rapids.enabled: true


In [4]:
def runMicroBenchmark(spark, appName, query, retryTimes):
    count = 0
    total_time = 0
    # You can print the physical plan of each query
    # spark.sql(query).explain()
    while count < retryTimes:
        start = time()
        spark.sql(query).show(5)
        end = time()
        total_time += round(end - start, 2)
        count = count + 1
        print("Retry times : {}, ".format(count) + appName + " microbenchmark takes {} seconds".format(round(end - start, 2)))
    print(appName + " microbenchmark takes average {} seconds after {} retries".format(round(total_time/retryTimes),retryTimes))
    with open('result.txt', 'a') as file:
        file.write("{},{},{}\n".format(appName, round(total_time/retryTimes), retryTimes))

In [5]:
dataRoot = "/users/chenqh23/spark-rapids-examples/datasets"

spark.read.parquet(dataRoot + "/tpcds/customer").createOrReplaceTempView("customer")
spark.read.parquet(dataRoot + "/tpcds/store_sales").createOrReplaceTempView("store_sales")
spark.read.parquet(dataRoot + "/tpcds/catalog_sales").createOrReplaceTempView("catalog_sales")
spark.read.parquet(dataRoot + "/tpcds/web_sales").createOrReplaceTempView("web_sales")
spark.read.parquet(dataRoot + "/tpcds/item").createOrReplaceTempView("item")
spark.read.parquet(dataRoot + "/tpcds/date_dim").createOrReplaceTempView("date_dim")
print("-"*50)

25/10/16 19:34:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


--------------------------------------------------


### Expand&HashAggregate
This is a microbenchmark about Expand&HashAggregate expressions running on the GPU. The query calculates the distinct value of some dimension columns and average birth year by different c_salutation of customers after grouping by c_current_hdemo_sk. You will see about 10x speedups in this query. Because an additional shuffle involved by the repartition operator in CPU mode. And GPUExpand and GPUHashAggregate is much faster than Expand and HashAggregate because GPU algorithms allow us to parallelize the computation and we can utilize most of the GPU cores. The tasks' duration in the third stage is less than one second but will cost 20x-40x while running on CPU. There will be a more significant performance improvement along with the increasing number of count distinct columns and aggregate functions.

In [6]:
query = '''
select c_current_hdemo_sk,
count(DISTINCT if(c_salutation=="Ms.",c_salutation,null)) as c1,
count(DISTINCT if(c_salutation=="Mr.",c_salutation,null)) as c12,
count(DISTINCT if(c_salutation=="Dr.",c_salutation,null)) as c13,

count(DISTINCT if(c_salutation=="Ms.",c_first_name,null)) as c2,
count(DISTINCT if(c_salutation=="Mr.",c_first_name,null)) as c22,
count(DISTINCT if(c_salutation=="Dr.",c_first_name,null)) as c23,

count(DISTINCT if(c_salutation=="Ms.",c_last_name,null)) as c3,
count(DISTINCT if(c_salutation=="Mr.",c_last_name,null)) as c32,
count(DISTINCT if(c_salutation=="Dr.",c_last_name,null)) as c33,

count(DISTINCT if(c_salutation=="Ms.",c_birth_country,null)) as c4,
count(DISTINCT if(c_salutation=="Mr.",c_birth_country,null)) as c42,
count(DISTINCT if(c_salutation=="Dr.",c_birth_country,null)) as c43,

count(DISTINCT if(c_salutation=="Ms.",c_email_address,null)) as c5,
count(DISTINCT if(c_salutation=="Mr.",c_email_address,null)) as c52,
count(DISTINCT if(c_salutation=="Dr.",c_email_address,null)) as c53,

count(DISTINCT if(c_salutation=="Ms.",c_login,null)) as c6,
count(DISTINCT if(c_salutation=="Mr.",c_login,null)) as c62,
count(DISTINCT if(c_salutation=="Dr.",c_login,null)) as c63,

count(DISTINCT if(c_salutation=="Ms.",c_preferred_cust_flag,null)) as c7,
count(DISTINCT if(c_salutation=="Mr.",c_preferred_cust_flag,null)) as c72,
count(DISTINCT if(c_salutation=="Dr.",c_preferred_cust_flag,null)) as c73,

count(DISTINCT if(c_salutation=="Ms.",c_birth_month,null)) as c8,
count(DISTINCT if(c_salutation=="Mr.",c_birth_month,null)) as c82,
count(DISTINCT if(c_salutation=="Dr.",c_birth_month,null)) as c83,

avg(if(c_salutation=="Ms.",c_birth_year,null)) as avg1,
avg(if(c_salutation=="Mr.",c_birth_year,null)) as avg2,
avg(if(c_salutation=="Dr.",c_birth_year,null)) as avg3,
avg(if(c_salutation=="Miss.",c_birth_year,null)) as avg4,
avg(if(c_salutation=="Mrs.",c_birth_year,null)) as avg5,
avg(if(c_salutation=="Sir.",c_birth_year,null)) as avg6,
avg(if(c_salutation=="Professor.",c_birth_year,null)) as avg7,
avg(if(c_salutation=="Teacher.",c_birth_year,null)) as avg8,
avg(if(c_salutation=="Agent.",c_birth_year,null)) as avg9,
avg(if(c_salutation=="Director.",c_birth_year,null)) as avg10
from customer group by c_current_hdemo_sk
'''

In [7]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Expand&HashAggregate",query,2)

25/10/16 19:34:18 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:34:18 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:34:18 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU bec

+------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------------------+------------------+------------------+----+------------------+----+----+----+----+-----+
|c_current_hdemo_sk| c1|c12|c13| c2|c22|c23| c3|c32|c33| c4|c42|c43| c5|c52|c53| c6|c62|c63| c7|c72|c73| c8|c82|c83|              avg1|              avg2|              avg3|avg4|              avg5|avg6|avg7|avg8|avg9|avg10|
+------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------------------+------------------+------------------+----+------------------+----+----+----+----+-----+
|                12|  1|  1|  1|  7| 14| 15|  7| 15| 17|  7| 15| 17|  7| 15| 17|  0|  0|  0|  2|  2|  2|  4|  9|  9|1948.7142857142858|1960.9333333333334|         1955.1875|NULL|          1954.875|NULL|NULL|NULL|NULL| NULL|
|                26|  1|  1|  1| 12| 11| 21| 12| 11| 22| 12| 10| 20| 12| 11| 21|  0|  0|  0|  2|  2|  2|

25/10/16 19:34:23 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:34:23 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:34:23 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU bec

+------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------------------+------------------+------------------+----+------------------+----+----+----+----+-----+
|c_current_hdemo_sk| c1|c12|c13| c2|c22|c23| c3|c32|c33| c4|c42|c43| c5|c52|c53| c6|c62|c63| c7|c72|c73| c8|c82|c83|              avg1|              avg2|              avg3|avg4|              avg5|avg6|avg7|avg8|avg9|avg10|
+------------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------------------+------------------+------------------+----+------------------+----+----+----+----+-----+
|                12|  1|  1|  1|  7| 14| 15|  7| 15| 17|  7| 15| 17|  7| 15| 17|  0|  0|  0|  2|  2|  2|  4|  9|  9|1948.7142857142858|1960.9333333333334|         1955.1875|NULL|          1954.875|NULL|NULL|NULL|NULL| NULL|
|                26|  1|  1|  1| 12| 11| 21| 12| 11| 22| 12| 10| 20| 12| 11| 21|  0|  0|  0|  2|  2|  2|

25/10/16 19:34:25 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:34:25 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU



### Windowing(without data skew)
This is a microbenchmark about windowing expressions running on GPU mode. The sub-query calculates the average ss_sales_price of a fixed window function partition by ss_customer_sk, and the parent query calculates the average price of the sub-query grouping by each customer. You will see about 25x speedups in this query. The speedup mainly comes from GPUSort/GPUWindow/GPUHashAggregate. The avg aggregation function evaluates all rows which are generated by the sub-query's window function. There will be a more significant performance improvement along with the increasing number of sub-query aggregate functions.

In [8]:
query = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
where ss_customer_sk is not null
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

--------------------------------------------------


In [9]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing without skew",query,2)

+--------------+-------------+
|ss_customer_sk|    avg_price|
+--------------+-------------+
|         56780|90.4028570000|
|         58111|90.0657140000|
|        142313|88.8262500000|
|        499355|88.7850000000|
|         74622|86.4722220000|
+--------------+-------------+
only showing top 5 rows

Retry times : 1, Windowing without skew microbenchmark takes 8.15 seconds


+--------------+-------------+
|ss_customer_sk|    avg_price|
+--------------+-------------+
|         56780|90.4028570000|
|         58111|90.0657140000|
|        142313|88.8262500000|
|        499355|88.7850000000|
|         74622|86.4722220000|
+--------------+-------------+
only showing top 5 rows

Retry times : 2, Windowing without skew microbenchmark takes 5.97 seconds
Windowing without skew microbenchmark takes average 7 seconds after 2 retries


### Windowing(with data skew)
Data skew is caused by many null values in the ss_customer_sk column. You will see about 80x speedups in this query. The heavier skew task a query has, the more improved performance we will get because GPU parallelizes the computation, CPU is limited to just a single core because of how the algorithms are written.

In [10]:
query = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

--------------------------------------------------


In [11]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing with skew",query,2)

+--------------+-------------+
|ss_customer_sk|    avg_price|
+--------------+-------------+
|         56780|90.4028570000|
|         58111|90.0657140000|
|        142313|88.8262500000|
|        499355|88.7850000000|
|         74622|86.4722220000|
+--------------+-------------+
only showing top 5 rows

Retry times : 1, Windowing with skew microbenchmark takes 6.36 seconds


+--------------+-------------+
|ss_customer_sk|    avg_price|
+--------------+-------------+
|         56780|90.4028570000|
|         58111|90.0657140000|
|        142313|88.8262500000|
|        499355|88.7850000000|
|         74622|86.4722220000|
+--------------+-------------+
only showing top 5 rows

Retry times : 2, Windowing with skew microbenchmark takes 5.76 seconds
Windowing with skew microbenchmark takes average 6 seconds after 2 retries


### Intersection
This is a microbenchmark about intersection operation running on GPU mode. The query calculates items in the same brand, class, and category that are sold in all three sales channels in two consecutive years. You will see about 10x speedups in this query. This is a competition between high cardinality SortMergeJoin vs GpuShuffleHashJoin. The mainly improved performance comes from two SortMergeJoin(s) in this query running on CPU get converted to GpuShuffleHashJoin running on GPU.

In [12]:
query = '''
select i_item_sk ss_item_sk
 from item,
    (select iss.i_brand_id brand_id, iss.i_class_id class_id, iss.i_category_id category_id
     from store_sales, item iss, date_dim d1
     where ss_item_sk = iss.i_item_sk
                    and ss_sold_date_sk = d1.d_date_sk
       and d1.d_year between 1999 AND 1999 + 2
   intersect
     select ics.i_brand_id, ics.i_class_id, ics.i_category_id
     from catalog_sales, item ics, date_dim d2
     where cs_item_sk = ics.i_item_sk
       and cs_sold_date_sk = d2.d_date_sk
       and d2.d_year between 1999 AND 1999 + 2
   intersect
     select iws.i_brand_id, iws.i_class_id, iws.i_category_id
     from web_sales, item iws, date_dim d3
     where ws_item_sk = iws.i_item_sk
       and ws_sold_date_sk = d3.d_date_sk
       and d3.d_year between 1999 AND 1999 + 2) x
 where i_brand_id = brand_id
   and i_class_id = class_id
   and i_category_id = category_id
'''

In [13]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"NDS Q14a subquery",query,2)

25/10/16 19:34:55 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:34:55 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:34:55 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU bec

+----------+
|ss_item_sk|
+----------+
|      8783|
|      2893|
|      4414|
|      4415|
|      7163|
+----------+
only showing top 5 rows

Retry times : 1, NDS Q14a subquery microbenchmark takes 11.18 seconds


25/10/16 19:35:06 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:35:06 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:35:06 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU bec

+----------+
|ss_item_sk|
+----------+
|     17316|
|     28534|
|      5019|
|      5021|
|     15224|
+----------+
only showing top 5 rows

Retry times : 2, NDS Q14a subquery microbenchmark takes 10.59 seconds
NDS Q14a subquery microbenchmark takes average 11 seconds after 2 retries


25/10/16 19:35:16 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU

25/10/16 19:35:16 WARN GpuOverrides: 
!Exec <CollectLimitExec> cannot run on GPU because the Exec CollectLimitExec has been disabled, and is disabled by default because Collect Limit replacement can be slower on the GPU, if huge number of rows in a batch it could help by limiting the number of rows transferred from GPU to CPU. Set spark.rapids.sql.exec.CollectLimitExec to true if you wish to enable it
  @Partitioning <SinglePartition$> could run on GPU



### Crossjoin
This is a microbenchmark for a 1-million rows crossjoin with itself. You will see about 10x speedups in this query. The mainly improved performance comes from converting BroadcastNestedLoogJoin running on CPU to GpuBroadcastNestedLoogJoin running on GPU.

In [14]:
start = time() 
spark.read.parquet(dataRoot + "/customer.dat").limit(1000000).write.format("parquet").mode("overwrite").save("/users/chenqh23/spark-rapids-examples/datasets/tmp/customer1m")
end = time()
# Parquet file scanning and writing will be about 3 times faster running on GPU
print("scanning and writing parquet cost : {} seconds".format(round(end - start, 2)))
spark.read.parquet("/users/chenqh23/spark-rapids-examples/datasets/tmp/customer1m").repartition(200).createOrReplaceTempView("costomer_df_1_million")
query = '''
select count(*) from costomer_df_1_million c1 inner join costomer_df_1_million c2 on c1.c_customer_sk>c2.c_customer_sk
'''
print("-"*50)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/users/chenqh23/spark-rapids-examples/datasets/customer.dat.

In [ ]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Crossjoin",query,2)

### HashJoin
This is a microbenchmark for a HashJoin. The query on GPU will be more than 10x times faster than CPU based on the cluster in the readme.

In [ ]:
spark.read.parquet(dataRoot + "/store_sales.dat").createOrReplaceTempView("store_sales")
spark.read.parquet(dataRoot + "/store_returns.dat").createOrReplaceTempView("store_returns")

print("-"*50)
query = '''
select  sum(store_sales.ss_ext_wholesale_cost)
from store_sales
join store_returns on (ss_item_sk = sr_item_sk) and (ss_addr_sk=sr_addr_sk)
'''
runMicroBenchmark(spark,"HashJoin",query,1)

In [15]:
spark.stop()